# Part 2: Retrieve evidence with microsoft-retriever-1

Cocoarynth's packaging team needs to know when a chocolate bar may carry the organic seal. Before asking a model to write an answer, find the policy passages that support it.

**Mission**
- Connect the existing policies index as an indexed knowledge source.
- Configure `microsoft-retriever-1` with the Python SDK and retrieve evidence without an answer-synthesis model.
- Inspect the actual activity and references returned by the service.

## The building blocks

A **knowledge source** connects an existing index to a **knowledge base**. This knowledge base uses `KnowledgeBaseMicrosoftRetriever1Model()`: there is no customer model endpoint or model key in that declaration.

Use `KnowledgeRetrievalMinimalReasoningEffort` and submit the complete question as one semantic **intent**. With `KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA`, the result is retrieved evidence, not a synthesized answer. Minimal alone does not establish MR1 execution: inspect the activity and planner metadata returned by the configured MR1 knowledge base.

This increment uses ordinary search indexes. For graph traversal, use the separate three-notebook graph lab. No graph calls or simulated graph results are included here.

## Pre-configured environment

The populated `lab673-preview-cocoarynth-policies` index is retained on the ramero dev stamp for these samples. Obtain the approved preview endpoint and key separately. The normal self-paced deployment is not assumed to enable this preview or populate a separate preview service.

These samples use the Azure Search Python SDK's native clients and models with its November preview default. Use the prepared kernel with the shared notebook requirements. Retrieval calls are billable. The notebook creates only notebook-prefixed knowledge sources and bases; it does not provision services, build indexes, or change billing settings.

## Step 1: Set up environment variables

Set `AZURE_SEARCH_PREVIEW_ENDPOINT`, `AZURE_SEARCH_PREVIEW_ADMIN_KEY`, and a unique `AZURE_SEARCH_PREVIEW_RESOURCE_PREFIX` starting with `lab673-`. See the [preview setup](../README.md#preview-sdk-notebooks) and the root `.env.sample`. The index prefix defaults to `lab673-preview-cocoarynth`; set `AZURE_SEARCH_PREVIEW_INDEX_PREFIX` only when using a different prepared dataset.

Open this notebook within the repository and select the existing lab Python kernel. Exported environment variables take precedence over the root `.env` file, so a local test endpoint does not get replaced by the normal deployment configuration. Never paste keys into cells or commit executed notebook outputs.

In [ ]:
import re
import importlib.metadata
from urllib.parse import urlsplit
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
import azure.search.documents as search_documents
from dotenv import load_dotenv
from azure.core import MatchConditions
import json
import os
from pathlib import Path

REPO_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / ".env.sample").is_file() and (directory / "src" / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

SDK_VERSION = "12.1.0a20261027002"
installed_version = importlib.metadata.version("azure-search-documents")
if installed_version != SDK_VERSION or search_documents.__version__ != SDK_VERSION:
    raise RuntimeError(
        f"Use azure-search-documents=={SDK_VERSION}; found installed {installed_version}, "
        f"imported {search_documents.__version__}. Restart the kernel after changing packages."
    )
load_dotenv(REPO_ROOT / ".env", override=False)
for setting in ('AZURE_SEARCH_PREVIEW_ENDPOINT', 'AZURE_SEARCH_PREVIEW_ADMIN_KEY', 'AZURE_SEARCH_PREVIEW_RESOURCE_PREFIX'):
    value = os.environ.get(setting, "").strip()
    if not value or "your-" in value:
        raise RuntimeError(f"Set {setting} in your environment or the root .env file.")

def check_response(response) -> None:
    status = response.http_response.status_code
    if status == 206:
        raise RuntimeError("Search returned HTTP 206: these results are incomplete.")
    if status < 400:
        return
    try:
        detail = json.loads(response.http_response.text())["error"]["message"]
    except (json.JSONDecodeError, KeyError, TypeError):
        detail = "The service did not return a JSON error message."
    if not isinstance(detail, str):
        detail = "The service did not return a text error message."
    for secret in (
        response.http_request.headers.get("api-key", ""),
        os.environ.get("AZURE_OPENAI_KEY", ""),
    ):
        if secret:
            detail = detail.replace(secret, "[REDACTED]")
    guidance = {
        402: " Ask the service owner about the Standard Knowledge Retrieval prerequisite.",
        412: " Another request changed this resource. Rerun the cell to read its current state.",
        429: " The service is busy or its allowance is exhausted. Retry later.",
    }.get(status, "")
    error_type = ResourceNotFoundError if status == 404 else HttpResponseError
    raise error_type(f"Search returned HTTP {status}: {detail}{guidance}") from None


CLIENT_OPTIONS = {
    "retry_total": 0,
    "permit_redirects": False,
    "connection_timeout": 15,
    "read_timeout": 120,
    "raw_response_hook": check_response,
}

RESOURCE_PREFIX = os.environ["AZURE_SEARCH_PREVIEW_RESOURCE_PREFIX"]
if not re.fullmatch(r"lab673-[a-z0-9](?:[a-z0-9-]{0,40}[a-z0-9])?", RESOURCE_PREFIX):
    raise ValueError("Use a unique lab673- resource prefix of at most 49 characters.")
OWNER_MARKER = f"LAB673 SDK notebooks: {RESOURCE_PREFIX} | "
ACCEPTED_OWNERS = (OWNER_MARKER, f"LAB673 REST notebooks: {RESOURCE_PREFIX} | ")
endpoint = os.environ["AZURE_SEARCH_PREVIEW_ENDPOINT"].rstrip("/")
origin = urlsplit(endpoint)
if (origin.scheme != "https" or not origin.hostname or origin.username or origin.password
        or origin.port not in (None, 443) or origin.path not in ("", "/")
        or origin.query or origin.fragment):
    raise ValueError("The Search endpoint must be an HTTPS service origin.")
search_credential = AzureKeyCredential(os.environ["AZURE_SEARCH_PREVIEW_ADMIN_KEY"])
index_client = SearchIndexClient(endpoint, search_credential, **CLIENT_OPTIONS)
INDEX_PREFIX = os.environ.get("AZURE_SEARCH_PREVIEW_INDEX_PREFIX", "lab673-preview-cocoarynth")
POLICIES_INDEX = f"{INDEX_PREFIX}-policies"
SOURCE_NAME = f"{RESOURCE_PREFIX}-policy-source"
KNOWLEDGE_BASE_NAME = f"{RESOURCE_PREFIX}-extractive"
print(f"Search SDK: {SDK_VERSION}; notebook resources: {RESOURCE_PREFIX}")

## Step 2: Verify the policies index

`SearchClient.get_document_count()` confirms that the prepared index contains document chunks. This notebook does not upload documents or generate embeddings.

In [ ]:
with SearchClient(endpoint, POLICIES_INDEX, search_credential, **CLIENT_OPTIONS) as search_client:
    count = search_client.get_document_count()
if not isinstance(count, int) or count <= 0:
    raise RuntimeError("The policies index must be populated before running this exercise.")
print(f"{POLICIES_INDEX}: {count} document chunks")

## Step 3: Create an indexed knowledge source

`SearchIndexKnowledgeSource` connects to the existing index. Search the `chunk` field and return the passage, document title, path, and page numbers so you can inspect the evidence. A knowledge-base semantic intent does not require a Search semantic query configuration.

The SDK calls below add a notebook ownership marker and use conditional writes. It refuses to overwrite a resource with a different owner. Use your own unique resource prefix, not another attendee's.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

source_definition = SearchIndexKnowledgeSource(
    name=SOURCE_NAME,
    description="Cocoarynth company policies, including label and marketing claims rules",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=POLICIES_INDEX,
        search_fields=[SearchIndexFieldReference(name="chunk")],
        source_data_fields=[
            SearchIndexFieldReference(name=name)
            for name in ("chunk", "blob_path", "title", "page_number_from", "page_number_to")
        ],
    ),
)
try:
    current = index_client.get_knowledge_source(source_definition.name)
except ResourceNotFoundError:
    current = None
if current is not None:
    if not (current.description or "").startswith(ACCEPTED_OWNERS):
        raise RuntimeError(f"Refusing to overwrite a resource owned by someone else: {source_definition.name}")
    if not current.e_tag:
        raise RuntimeError(f"Cannot safely update a resource without its ETag: {source_definition.name}")
source_definition.description = OWNER_MARKER + (source_definition.description or "")
source_definition.e_tag = current.e_tag if current is not None else None
condition = MatchConditions.IfNotModified if current is not None else MatchConditions.IfMissing
index_client.create_or_update_knowledge_source(source_definition, match_condition=condition)
print(f"Knowledge source ready: {SOURCE_NAME}")

## Step 4: Create the retriever knowledge base

`KnowledgeBaseMicrosoftRetriever1Model()` selects the built-in retriever and extractive output. This step does not need an Azure OpenAI deployment or key. Any model configuration already used by the prepared index is separate from this knowledge base's model declaration.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_base_definition = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Retrieve Cocoarynth policy evidence without answer synthesis",
    knowledge_sources=[KnowledgeSourceReference(name=SOURCE_NAME)],
    models=[KnowledgeBaseMicrosoftRetriever1Model()],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
try:
    current = index_client.get_knowledge_base(knowledge_base_definition.name)
except ResourceNotFoundError:
    current = None
if current is not None:
    if not (current.description or "").startswith(ACCEPTED_OWNERS):
        raise RuntimeError(f"Refusing to overwrite a resource owned by someone else: {knowledge_base_definition.name}")
    if not current.e_tag:
        raise RuntimeError(f"Cannot safely update a resource without its ETag: {knowledge_base_definition.name}")
knowledge_base_definition.description = OWNER_MARKER + (knowledge_base_definition.description or "")
knowledge_base_definition.e_tag = current.e_tag if current is not None else None
condition = MatchConditions.IfNotModified if current is not None else MatchConditions.IfMissing
index_client.create_or_update_knowledge_base(knowledge_base_definition, match_condition=condition)
print(f"Knowledge base ready: {KNOWLEDGE_BASE_NAME}")

## Step 5: Retrieve the policy evidence

`KnowledgeBaseRetrievalClient.retrieve()` accepts a semantic intent rather than a chat message for this Minimal request. Ask for references, their source data, and the activity log explicitly.

This is a non-streaming request. `max_runtime_in_seconds` bounds service work; `max_output_documents` bounds returned evidence. Use `max_output_size=6000` for this preview request.

If the service reports HTTP 402 requiring Standard Knowledge Retrieval, ask the service owner to resolve that prerequisite. Do not treat an error or missing references as a completed exercise.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)

QUESTION = (
    "When may a Cocoarynth bar carry the organic seal, and what if the supplier's "
    "organic certificate is pending renewal?"
)
retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint, search_credential, knowledge_base_name=KNOWLEDGE_BASE_NAME, **CLIENT_OPTIONS
)
retrieval_request = KnowledgeBaseRetrievalRequest(
    intents=[KnowledgeRetrievalSemanticIntent(search=QUESTION)],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    include_activity=True,
    knowledge_source_params=[SearchIndexKnowledgeSourceParams(
        knowledge_source_name=SOURCE_NAME,
        include_references=True,
        include_reference_source_data=True,
        always_query_source=True,
    )],
    max_runtime_in_seconds=60,
    max_output_documents=8,
    max_output_size=6000,
)
print(json.dumps(retrieval_request.as_dict(), indent=2))
result = retrieval_client.retrieve(retrieval_request)
if not result.references:
    raise RuntimeError("No references returned. Check the prepared index and source configuration.")
print(f"Retrieved {len(result.references)} references; no answer synthesis was requested.")

### Review the activity log

Inspect the returned `type`, `id`, model metadata, query text, and timing where present. The sequence and number of steps can vary with the question and service response. Activity is evidence of execution, not a guarantee that every relevant passage was found.

In [ ]:
if not result.activity:
    raise RuntimeError("The service did not return the requested activity log.")
print(json.dumps([activity.as_dict() for activity in result.activity], indent=2))

### Review the references

`sourceData` contains the fields requested on the knowledge source. Use `activitySource` to connect a reference to its retrieval activity when that field is present. Keep the service's reference IDs; do not renumber them.

### Source hunt

Find the policy passage that states the certificate requirement on the packing date. Does it also explain what a pending renewal means for the seal? Which document and page support your conclusion?

In [ ]:
print(json.dumps([reference.as_dict() for reference in result.references], indent=2))

## Optional: Clean up notebook resources

Leave `CLEAN_UP` false while exploring. Set it to true and run the next cell when finished. It deletes only this notebook's knowledge base and source, in that order. The prepared index and Part 1 resources remain unchanged.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    targets = [
        (KNOWLEDGE_BASE_NAME, index_client.get_knowledge_base, index_client.delete_knowledge_base),
        *((name, index_client.get_knowledge_source, index_client.delete_knowledge_source)
          for name in [SOURCE_NAME]),
    ]
    for name, get_resource, delete_resource in targets:
        try:
            current = get_resource(name)
        except ResourceNotFoundError:
            print(f"Already absent: {name}")
            continue
        if not (current.description or "").startswith(ACCEPTED_OWNERS):
            raise RuntimeError(f"Refusing to delete a resource owned by someone else: {name}")
        if not current.e_tag:
            raise RuntimeError(f"Cannot safely delete a resource without its ETag: {name}")
        delete_resource(current, match_condition=MatchConditions.IfNotModified)
        try:
            get_resource(name)
        except ResourceNotFoundError:
            print(f"Removed: {name}")
        else:
            raise RuntimeError(f"The resource is still present after deletion: {name}")
    print("This notebook's resources are removed; prepared indexes are unchanged.")
else:
    print("Resources retained. Set CLEAN_UP to True when you finish exploring.")

## Mission complete

You configured a built-in retriever with the Python SDK, retrieved policy evidence, and inspected its activity and references without asking for a generated answer.

Continue to [Part 3: Add answer synthesis to the retriever](part3-retriever-answer-synthesis.ipynb) to investigate a multi-document mystery.